# DDPM 扩散模型 — MNIST 手写数字生成

## 理论背景

**扩散模型（Diffusion Model）** 的核心思想来自非平衡态热力学：

1. **前向过程（Forward Process）**：逐步向真实图像中添加高斯噪声，经过 $T$ 步后变成纯噪声
2. **反向过程（Reverse Process）**：训练神经网络从噪声中逐步恢复出原始图像

### 关键公式

**前向扩散（闭式解，一步到位）**：
$$x_t = \sqrt{\bar{\alpha}_t} \cdot x_0 + \sqrt{1 - \bar{\alpha}_t} \cdot \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

**训练目标（预测噪声）**：
$$\mathcal{L} = \mathbb{E}_{x_0, t, \epsilon}\left[ \|\epsilon - \epsilon_\theta(x_t, t)\|^2 \right]$$

**反向采样（DDPM）**：
$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}} \left(x_t - \frac{\beta_t}{\sqrt{1-\bar{\alpha}_t}} \epsilon_\theta(x_t, t) \right) + \sigma_t \cdot z$$

---

## 本 Notebook 结构

| Section | 内容 |
|---------|------|
| 1 | 依赖导入 |
| 2 | MNIST 数据准备 |
| 3 | 前向扩散过程 + 可视化 |
| 4 | **U-Net 去噪网络**（核心架构）|
| 5 | 训练循环 |
| 6 | DDPM 采样生成 |
| 7 | DDIM 加速采样（加分项）|
| 8 | 潜在空间插值 + 总结 |

In [ ]:
import math
from pathlib import Path
from copy import deepcopy

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import datasets, transforms
from torchvision.utils import make_grid, save_image

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# ── 全局配置 ──────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"PyTorch 版本: {torch.__version__}")
print(f"计算设备  : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU 型号  : {torch.cuda.get_device_name(0)}")
    print(f"显存      : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

---
## Section 2 · MNIST 数据准备

使用 `torchvision.datasets.MNIST` 自动下载，将像素归一化到 `[-1, 1]`（扩散模型标准做法）。

In [ ]:
# ── 数据加载 ──────────────────────────────────────────
BATCH_SIZE = 128

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),  # [0,1] → [-1,1]
])

train_dataset = datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=0, pin_memory=True, drop_last=True,
)

print(f"训练样本数: {len(train_dataset)}")
print(f"批次数量  : {len(train_loader)}")
print(f"图像尺寸  : 28×28 灰度图")

In [ ]:
# ── 快速可视化 ────────────────────────────────────────
imgs, labels = next(iter(train_loader))
# 反归一化: [-1,1] → [0,1]
imgs_show = imgs * 0.5 + 0.5
grid = make_grid(imgs_show[:16], nrow=4)

plt.figure(figsize=(6, 6))
plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray")
plt.title("MNIST 样本（前 16 张）", fontsize=14)
plt.axis("off")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "mnist_samples.png", dpi=100, bbox_inches="tight")
plt.show()

---
## Section 3 · 前向扩散过程

定义 **噪声调度（Noise Schedule）** 和 **前向加噪函数**。

- $\beta_t$：线性从 $10^{-4}$ 到 $0.02$
- $\alpha_t = 1 - \beta_t$
- $\bar{\alpha}_t = \prod_{s=1}^{t} \alpha_s$（累积乘积）

前向扩散使用**闭式解**，不需要逐步迭代：
$$x_t = \sqrt{\bar{\alpha}_t} \cdot x_0 + \sqrt{1 - \bar{\alpha}_t} \cdot \epsilon$$

In [ ]:
# ── 噪声调度 ──────────────────────────────────────────
T = 1000  # 扩散总步数

betas = torch.linspace(1e-4, 0.02, T)           # 线性 schedule
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)    # ᾱ_t

# 预计算前向扩散所需系数
sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)

# 预计算反向采样所需系数
sqrt_recip_alphas = torch.sqrt(1.0 / alphas)
alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value=1.0)
posterior_variance = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)

print(f"扩散总步数 T = {T}")
print(f"β range      : [{betas[0]:.4f}, {betas[-1]:.4f}]")
print(f"ᾱ_T          : {alphas_cumprod[-1]:.6f}  (≈0，即终态接近纯噪声)")

# ── 前向扩散（闭式解） ────────────────────────────────
def q_sample(x_0: torch.Tensor, t: torch.Tensor, noise: torch.Tensor = None):
    """
    从 x_0 一步加噪到 x_t。
    
    Args:
        x_0:   原始图像 (B, C, H, W)
        t:     时间步 (B,)
        noise: 预设噪声，None 则随机生成
    Returns:
        x_t: 加噪后的图像
    """
    if noise is None:
        noise = torch.randn_like(x_0)
    
    sqrt_alpha_t = sqrt_alphas_cumprod[t.cpu()].to(x_0.device)
    sqrt_one_minus_alpha_t = sqrt_one_minus_alphas_cumprod[t.cpu()].to(x_0.device)
    
    # 调整维度以广播: (B,) → (B, 1, 1, 1)
    while sqrt_alpha_t.dim() < x_0.dim():
        sqrt_alpha_t = sqrt_alpha_t.unsqueeze(-1)
        sqrt_one_minus_alpha_t = sqrt_one_minus_alpha_t.unsqueeze(-1)
    
    return sqrt_alpha_t * x_0 + sqrt_one_minus_alpha_t * noise

In [ ]:
# ── 可视化前向扩散过程 ────────────────────────────────
def visualize_forward_diffusion(x_0, steps=None):
    """展示一张图在扩散过程中如何逐渐变成噪声"""
    if steps is None:
        steps = [0, 50, 100, 200, 300, 400, 500, 600, 700, 800, 900, 999]
    
    fig, axes = plt.subplots(1, len(steps), figsize=(16, 2))
    for i, t_val in enumerate(steps):
        t_tensor = torch.full((1,), t_val, dtype=torch.long)
        x_t = q_sample(x_0.unsqueeze(0), t_tensor)
        img = x_t.squeeze() * 0.5 + 0.5  # 反归一化
        axes[i].imshow(img, cmap="gray")
        axes[i].set_title(f"t={t_val}" if t_val > 0 else "t=0 (原始)", fontsize=9)
        axes[i].axis("off")
    
    plt.suptitle("前向扩散过程：图像 → 纯噪声", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "forward_diffusion.png", dpi=100, bbox_inches="tight")
    plt.show()

# 取一张样本演示
sample_img, _ = train_dataset[0]
visualize_forward_diffusion(sample_img)

In [ ]:
# ── β schedule 曲线 ──────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(betas.numpy())
ax1.set_xlabel("Timestep t")
ax1.set_ylabel("β_t")
ax1.set_title("Linear Noise Schedule β_t")
ax1.grid(True, alpha=0.3)

ax2.plot(alphas_cumprod.numpy())
ax2.set_xlabel("Timestep t")
ax2.set_ylabel("ᾱ_t")
ax2.set_title("Cumulative Product ᾱ_t (信噪比)")
ax2.grid(True, alpha=0.3)
ax2.axhline(y=0, color="red", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "noise_schedule.png", dpi=100, bbox_inches="tight")
plt.show()

---
## Section 4 · U-Net 去噪网络（核心架构）

U-Net 是一个**编码器-解码器**架构，带跳跃连接。扩散模型中的 U-Net 额外加入了：

1. **时间嵌入（Time Embedding）**：让网络知道当前处于扩散的哪一步
2. **自注意力（Self-Attention）**：在低分辨率层捕获全局结构

### 架构图

```
Input (1,28,28) ─────────────────────────────┐
  ↓ Conv In (64ch)                           │
  ├─ ResBlock ×2 + SelfAttn (64ch, 28×28) ───┤ skip → concat
  ↓ Downsample (128ch, 14×14)                │
  ├─ ResBlock ×2 + SelfAttn (128ch, 14×14) ──┤ skip → concat
  ↓ Downsample (256ch, 7×7)                  │
  ├─ ResBlock (256ch, 7×7)                   │
  ├─ SelfAttention (256ch, 7×7)  ← 全局注意力  │
  ├─ ResBlock (256ch, 7×7)                   │
  ↓ Upsample (128ch, 14×14)                  │
  ├─ Concat + ResBlock ×2 (256ch→128ch)      │
  ↓ Upsample (64ch, 28×28)                   │
  ├─ Concat + ResBlock ×2 (128ch→64ch)       │
  ↓ Conv Out (1ch, 28×28)
Output (1,28,28) ← 预测的噪声
```

In [ ]:
# ═══════════════════════════════════════════════════════
# 组件 1: Sinusoidal 时间位置编码
# ═══════════════════════════════════════════════════════

class SinusoidalPosEmb(nn.Module):
    """
    Transformer 风格的正弦位置编码。
    将标量时间步 t ∈ [0, T) 编码为高维向量，
    不同频率的正弦/余弦捕获不同尺度的时序信息。
    """
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        """
        Args:
            t: 时间步索引 (B,)
        Returns:
            时间嵌入向量 (B, dim)
        """
        device = t.device
        half_dim = self.dim // 2
        # 频率: 1 / (10000^(2i/dim))
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = t[:, None].float() * emb[None, :]       # (B, half_dim)
        emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)
        return emb

In [ ]:
# ═══════════════════════════════════════════════════════
# 组件 2: 残差块（Residual Block）
# ═══════════════════════════════════════════════════════

class ResBlock(nn.Module):
    """
    含时间条件注入的残差块。
    
    结构：GroupNorm → SiLU → Conv → +TimeProj → GroupNorm → SiLU → Conv → +Residual
    
    使用 GroupNorm 而非 BatchNorm：扩散模型对小批量统计量敏感，
    GroupNorm 独立于 batch size，更稳定。
    """
    def __init__(self, in_ch: int, out_ch: int, time_emb_dim: int, groups: int = 8):
        super().__init__()
        self.norm1 = nn.GroupNorm(min(groups, in_ch), in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1)
        
        self.norm2 = nn.GroupNorm(min(groups, out_ch), out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1)
        
        # 将时间嵌入映射到通道维度
        self.time_proj = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_emb_dim, out_ch),
        )
        
        # 1×1 卷积做通道对齐（维度不匹配时）
        self.res_conv = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x: torch.Tensor, t_emb: torch.Tensor) -> torch.Tensor:
        h = self.norm1(x)
        h = F.silu(h)
        h = self.conv1(h)
        # 注入时间条件：逐通道缩放
        h = h + self.time_proj(t_emb)[:, :, None, None]
        
        h = self.norm2(h)
        h = F.silu(h)
        h = self.conv2(h)
        
        return h + self.res_conv(x)

In [ ]:
# ═══════════════════════════════════════════════════════
# 组件 3: 自注意力层（Self-Attention）
# ═══════════════════════════════════════════════════════

class SelfAttention(nn.Module):
    """
    多头自注意力，在低分辨率特征图上捕获全局依赖。
    
    为何在低分辨率层加注意力？
    - 高分辨率（28×28=784 tokens）：计算量大，局部纹理已够
    - 低分辨率（7×7=49 tokens）：计算量小，全局结构重要
    
    使用 GroupNorm(groups=1) 等价于 LayerNorm，适合 transformer 风格。
    """
    def __init__(self, channels: int, num_heads: int = 4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = channels // num_heads
        self.scale = self.head_dim ** -0.5
        
        self.norm = nn.GroupNorm(1, channels)  # GroupNorm(1)=LayerNorm
        self.to_qkv = nn.Conv2d(channels, channels * 3, kernel_size=1)
        self.to_out = nn.Conv2d(channels, channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        
        h = self.norm(x)
        # (B, 3C, H, W) → (B, 3, heads, head_dim, H*W)
        qkv = self.to_qkv(h).reshape(B, 3, self.num_heads, self.head_dim, H * W)
        q, k, v = qkv[:, 0], qkv[:, 1], qkv[:, 2]
        
        # 缩放点积注意力: softmax(QK^T / √d) · V
        attn = torch.einsum("bhdn,bhdm->bhnm", q, k) * self.scale
        attn = F.softmax(attn, dim=-1)
        
        out = torch.einsum("bhnm,bhdm->bhdn", attn, v)
        out = out.reshape(B, C, H, W)
        return self.to_out(out) + x  # 残差连接

In [ ]:
# ═══════════════════════════════════════════════════════
# 组件 4: 完整 U-Net（显式架构，通道精确对齐）
# ═══════════════════════════════════════════════════════

class UNet(nn.Module):
    """
    扩散模型去噪 U-Net，专为 MNIST 28×28 设计。
    
    架构（3 级编码器 → 瓶颈 → 3 级解码器）:
    
    Input (B, 1, 28, 28)
      ├─ Conv In  →  (B, 64, 28, 28)
      ├─ Enc1: ResBlock×2  →  (B, 64, 28, 28)  ── skip1
      ├─ Down1: Conv stride2  →  (B, 128, 14, 14)
      ├─ Enc2: ResBlock×2 + SelfAttn  →  (B, 128, 14, 14)  ── skip2
      ├─ Down2: Conv stride2  →  (B, 256, 7, 7)
      ├─ Mid: ResBlock + SelfAttn + ResBlock  →  (B, 256, 7, 7)
      ├─ Up1: Upsample + Conv  →  (B, 128, 14, 14)
      ├─ Concat(skip2)  →  (B, 256, 14, 14)
      ├─ Dec2: ResBlock×3 + SelfAttn  →  (B, 128, 14, 14)
      ├─ Up2: Upsample + Conv  →  (B, 64, 28, 28)
      ├─ Concat(skip1)  →  (B, 128, 28, 28)
      ├─ Dec1: ResBlock×3  →  (B, 64, 28, 28)
      └─ Conv Out  →  (B, 1, 28, 28)
    
    Args:
        in_ch:        输入通道数（MNIST 灰度 = 1）
        base_ch:      基础通道数
        time_emb_dim: 时间嵌入维度
    """
    def __init__(
        self,
        in_ch: int = 1,
        base_ch: int = 64,
        time_emb_dim: int = 128,
    ):
        super().__init__()
        
        # ── 时间嵌入 MLP ──
        self.time_mlp = nn.Sequential(
            SinusoidalPosEmb(time_emb_dim),
            nn.Linear(time_emb_dim, time_emb_dim * 4),
            nn.SiLU(),
            nn.Linear(time_emb_dim * 4, time_emb_dim),
        )
        
        # ── 输入卷积 ──
        self.conv_in = nn.Conv2d(in_ch, base_ch, kernel_size=3, padding=1)
        
        # ── 编码器 Level 1: 28×28, 64ch ──
        self.enc1_block1 = ResBlock(base_ch, base_ch, time_emb_dim)      # 64→64
        self.enc1_block2 = ResBlock(base_ch, base_ch, time_emb_dim)      # 64→64
        self.down1 = nn.Conv2d(base_ch, base_ch * 2, 3, stride=2, padding=1)  # 64→128, 28→14
        
        # ── 编码器 Level 2: 14×14, 128ch ──
        ch2 = base_ch * 2  # 128
        self.enc2_block1 = ResBlock(ch2, ch2, time_emb_dim)              # 128→128
        self.enc2_block2 = ResBlock(ch2, ch2, time_emb_dim)              # 128→128
        self.enc2_attn = SelfAttention(ch2)
        self.down2 = nn.Conv2d(ch2, ch2 * 2, 3, stride=2, padding=1)    # 128→256, 14→7
        
        # ── 瓶颈层: 7×7, 256ch ──
        mid_ch = base_ch * 4  # 256
        self.mid_block1 = ResBlock(mid_ch, mid_ch, time_emb_dim)
        self.mid_attn = SelfAttention(mid_ch)
        self.mid_block2 = ResBlock(mid_ch, mid_ch, time_emb_dim)
        
        # ── 解码器 Level 2: 7→14, 256→128ch ──
        self.up1 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(mid_ch, ch2, 3, padding=1),  # 256 → 128
        )
        # concat(up1_out=128 + skip2=128) = 256 input
        self.dec2_block1 = ResBlock(ch2 * 2, ch2, time_emb_dim)          # 256→128
        self.dec2_block2 = ResBlock(ch2, ch2, time_emb_dim)              # 128→128
        self.dec2_block3 = ResBlock(ch2, ch2, time_emb_dim)              # 128→128
        self.dec2_attn = SelfAttention(ch2)
        
        # ── 解码器 Level 1: 14→28, 128→64ch ──
        self.up2 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(ch2, base_ch, 3, padding=1),  # 128 → 64
        )
        # concat(up2_out=64 + skip1=64) = 128 input
        self.dec1_block1 = ResBlock(base_ch * 2, base_ch, time_emb_dim)  # 128→64
        self.dec1_block2 = ResBlock(base_ch, base_ch, time_emb_dim)      # 64→64
        self.dec1_block3 = ResBlock(base_ch, base_ch, time_emb_dim)      # 64→64
        
        # ── 输出卷积 ──
        self.conv_out = nn.Sequential(
            nn.GroupNorm(min(8, base_ch), base_ch),
            nn.SiLU(),
            nn.Conv2d(base_ch, in_ch, kernel_size=3, padding=1),
        )
        
        self._init_weights()
    
    def _init_weights(self):
        """Kaiming 初始化 + 输出层零初始化"""
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.kaiming_normal_(m.weight, mode="fan_in", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
        nn.init.zeros_(self.conv_out[-1].weight)
        nn.init.zeros_(self.conv_out[-1].bias)

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: 加噪图像 x_t，(B, C, H, W)
            t: 时间步索引 (B,)
        Returns:
            预测的噪声 ε_θ(x_t, t)，同形状
        """
        t_emb = self.time_mlp(t)
        
        # ── 编码器 ──
        h = self.conv_in(x)                         # (B, 64, 28, 28)
        
        h = self.enc1_block1(h, t_emb)              # (B, 64, 28, 28)
        h = self.enc1_block2(h, t_emb)              # (B, 64, 28, 28)
        skip1 = h                                    # 保存跳跃连接
        h = self.down1(h)                            # (B, 128, 14, 14)
        
        h = self.enc2_block1(h, t_emb)              # (B, 128, 14, 14)
        h = self.enc2_block2(h, t_emb)              # (B, 128, 14, 14)
        h = self.enc2_attn(h)                        # (B, 128, 14, 14)
        skip2 = h                                    # 保存跳跃连接
        h = self.down2(h)                            # (B, 256, 7, 7)
        
        # ── 瓶颈 ──
        h = self.mid_block1(h, t_emb)               # (B, 256, 7, 7)
        h = self.mid_attn(h)                         # (B, 256, 7, 7)
        h = self.mid_block2(h, t_emb)               # (B, 256, 7, 7)
        
        # ── 解码器 ──
        h = self.up1(h)                              # (B, 128, 14, 14)
        h = torch.cat([h, skip2], dim=1)            # (B, 256, 14, 14)
        h = self.dec2_block1(h, t_emb)              # (B, 128, 14, 14)
        h = self.dec2_block2(h, t_emb)              # (B, 128, 14, 14)
        h = self.dec2_block3(h, t_emb)              # (B, 128, 14, 14)
        h = self.dec2_attn(h)                        # (B, 128, 14, 14)
        
        h = self.up2(h)                              # (B, 64, 28, 28)
        h = torch.cat([h, skip1], dim=1)            # (B, 128, 28, 28)
        h = self.dec1_block1(h, t_emb)              # (B, 64, 28, 28)
        h = self.dec1_block2(h, t_emb)              # (B, 64, 28, 28)
        h = self.dec1_block3(h, t_emb)              # (B, 64, 28, 28)
        
        return self.conv_out(h)                      # (B, 1, 28, 28)


# ── 实例化 + 参数量统计 ──
model = UNet().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"U-Net 参数量: {total_params:,} (可训练: {trainable_params:,})")

# 快速测试前向传播
with torch.no_grad():
    test_x = torch.randn(4, 1, 28, 28).to(DEVICE)
    test_t = torch.randint(0, T, (4,)).to(DEVICE)
    test_out = model(test_x, test_t)
    print(f"输入形状: {test_x.shape} → 输出形状: {test_out.shape} ✓")
    print(f"U-Net 通道流: 1 → 64 → 128 → 256 → 128 → 64 → 1")

---
## Section 5 · 训练循环

训练流程极其简洁——和普通图像分类一样：

1. 随机采样真实图像 $x_0$
2. 随机采样时间步 $t \sim \text{Uniform}(0, T)$
3. 随机采样噪声 $\epsilon \sim \mathcal{N}(0, I)$
4. 前向加噪: $x_t = \sqrt{\bar{\alpha}_t} \cdot x_0 + \sqrt{1-\bar{\alpha}_t} \cdot \epsilon$
5. 模型预测: $\hat{\epsilon} = \epsilon_\theta(x_t, t)$
6. 计算 Loss: $\mathcal{L} = \text{MSE}(\epsilon, \hat{\epsilon})$

**额外技巧**：
- **EMA（指数移动平均）**：维护一份平滑的模型副本用于最终采样
- **混合精度训练**：`autocast` + `GradScaler` 加速训练

In [ ]:
# ── 训练超参数 ────────────────────────────────────────
EPOCHS = 50
LR = 1e-3
EMA_DECAY = 0.995
EMA_UPDATE_AFTER = 100  # 前 100 步不更新 EMA（等参数稳定）

# 优化器 + 调度器
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

# 混合精度
scaler = torch.amp.GradScaler("cuda") if DEVICE.type == "cuda" else None

# EMA 模型（用于生成，平滑后质量更好）
ema_model = deepcopy(model)
ema_model.eval()

print(f"训练轮数  : {EPOCHS}")
print(f"学习率    : {LR}")
print(f"EMA 衰减  : {EMA_DECAY}")
print(f"批次大小  : {BATCH_SIZE}")
print(f"总训练步数: {EPOCHS * len(train_loader)}")

In [ ]:
# ── 训练循环 ──────────────────────────────────────────
loss_history = []
global_step = 0

model.train()
print(f"{'Epoch':>6} {'Loss':>10} {'LR':>10}  进度")
print("-" * 55)

for epoch in range(1, EPOCHS + 1):
    epoch_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch:2d}/{EPOCHS}", leave=False)
    
    for imgs, _ in pbar:
        imgs = imgs.to(DEVICE)  # (B, 1, 28, 28)
        B = imgs.shape[0]
        
        optimizer.zero_grad()
        
        if scaler:
            with torch.amp.autocast("cuda"):
                # 1. 随机采样时间步和噪声
                t = torch.randint(0, T, (B,), device=DEVICE, dtype=torch.long)
                noise = torch.randn_like(imgs)
                
                # 2. 前向加噪（闭式解）
                x_t = q_sample(imgs, t, noise)
                
                # 3. 预测噪声
                noise_pred = model(x_t, t)
                
                # 4. MSE 损失
                loss = F.mse_loss(noise_pred, noise)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            t = torch.randint(0, T, (B,), device=DEVICE, dtype=torch.long)
            noise = torch.randn_like(imgs)
            x_t = q_sample(imgs, t, noise)
            noise_pred = model(x_t, t)
            loss = F.mse_loss(noise_pred, noise)
            loss.backward()
            optimizer.step()
        
        # ── EMA 更新 ──
        if global_step >= EMA_UPDATE_AFTER:
            with torch.no_grad():
                for ema_p, p in zip(ema_model.parameters(), model.parameters()):
                    ema_p.data = EMA_DECAY * ema_p.data + (1 - EMA_DECAY) * p.data
        
        epoch_loss += loss.item()
        global_step += 1
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    
    avg_loss = epoch_loss / len(train_loader)
    loss_history.append(avg_loss)
    scheduler.step()
    lr_now = scheduler.get_last_lr()[0]
    
    print(f"{epoch:>5d}  {avg_loss:>10.4f}  {lr_now:>10.2e}  {'█' * (epoch * 30 // EPOCHS)}")

print(f"\n训练完成！总步数: {global_step}")

In [ ]:
# ── Loss 曲线 ─────────────────────────────────────────
plt.figure(figsize=(10, 4))
plt.plot(loss_history, marker="o", markersize=3, linewidth=1)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("训练 Loss 曲线")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "loss_curve.png", dpi=100, bbox_inches="tight")
plt.show()

print(f"初始 Loss: {loss_history[0]:.4f}")
print(f"最终 Loss: {loss_history[-1]:.4f}")
print(f"下降幅度: {(1 - loss_history[-1] / loss_history[0]) * 100:.1f}%")

In [ ]:
# ── 保存模型权重 ──────────────────────────────────────
checkpoint = {
    "model": model.state_dict(),
    "ema_model": ema_model.state_dict(),
    "optimizer": optimizer.state_dict(),
    "epoch": EPOCHS,
    "loss_history": loss_history,
    "config": {
        "T": T,
        "base_ch": 64,
    },
}
torch.save(checkpoint, OUTPUT_DIR / "ddpm_mnist.pth")
print(f"模型已保存至: {OUTPUT_DIR / 'ddpm_mnist.pth'}")

---
## Section 6 · DDPM 采样生成

从纯噪声出发，逐步去噪 $T$ 步，最终得到清晰的生成图像。

**DDPM 反向采样公式**：
$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}} \left( x_t - \frac{\beta_t}{\sqrt{1-\bar{\alpha}_t}} \cdot \epsilon_\theta(x_t, t) \right) + \sigma_t \cdot z$$

其中 $z \sim \mathcal{N}(0, I)$（$t>1$ 时），$t=1$ 时不加噪声。

In [ ]:
# ── DDPM 采样 ─────────────────────────────────────────
@torch.no_grad()
def ddpm_sample(model, shape, return_trajectory=False):
    """
    DDPM 反向采样：纯噪声 → 逐步去噪 → 清晰图像。
    
    Args:
        model: U-Net 去噪网络
        shape: 生成图像的形状 (B, C, H, W)
        return_trajectory: 是否返回去噪过程的所有中间态
    Returns:
        生成的图像 (B, C, H, W)，值域 [-1, 1]
    """
    model.eval()
    
    # 从纯噪声开始
    x = torch.randn(shape, device=DEVICE)
    trajectory = [x.cpu()] if return_trajectory else None
    
    # T → 1 逐步去噪
    for t in tqdm(reversed(range(T)), desc="DDPM Sampling", total=T, leave=False):
        t_batch = torch.full((shape[0],), t, device=DEVICE, dtype=torch.long)
        
        # 预测噪声
        noise_pred = model(x, t_batch)
        
        # DDPM 反向更新
        alpha_t = alphas[t].to(DEVICE)
        alpha_cumprod_t = alphas_cumprod[t].to(DEVICE)
        beta_t = betas[t].to(DEVICE)
        
        # 均值: 1/√α_t · (x_t - β_t/√(1-ᾱ_t) · ε_θ)
        x = sqrt_recip_alphas[t].to(DEVICE) * (
            x - beta_t / sqrt_one_minus_alphas_cumprod[t].to(DEVICE) * noise_pred
        )
        
        # 方差: σ_t · z（t > 0 时加噪声）
        if t > 0:
            noise = torch.randn_like(x)
            sigma_t = torch.sqrt(posterior_variance[t]).to(DEVICE)
            x = x + sigma_t * noise
        
        if return_trajectory and t % 100 == 0:
            trajectory.append(x.cpu())
    
    if return_trajectory:
        trajectory.append(x.cpu())
        return x, trajectory
    return x


print("DDPM 采样函数已定义")
print(f"从纯噪声开始，经 {T} 步去噪 → 最终图像")

In [ ]:
# ── 生成并可视化 ──────────────────────────────────────
N_SAMPLES = 16

print("正在生成（使用 EMA 模型，约需 1-2 分钟）...")
samples = ddpm_sample(ema_model, shape=(N_SAMPLES, 1, 28, 28))

# 反归一化 + 拼接
samples_show = samples * 0.5 + 0.5
samples_show = torch.clamp(samples_show, 0, 1)
grid = make_grid(samples_show, nrow=4, padding=2)

# 显示
plt.figure(figsize=(8, 8))
plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray")
plt.title(f"DDPM 生成结果（{N_SAMPLES} 张，{T} 步采样）", fontsize=14)
plt.axis("off")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "ddpm_samples.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── 去噪过程动画 ──────────────────────────────────────
print("生成去噪过程...")
_, trajectory = ddpm_sample(ema_model, shape=(1, 1, 28, 28), return_trajectory=True)

fig, axes = plt.subplots(1, len(trajectory), figsize=(16, 2.5))
steps_shown = list(range(T, -1, -100))
for i, (step, img) in enumerate(zip(steps_shown, trajectory)):
    ax = axes[i]
    img_show = img.squeeze() * 0.5 + 0.5
    ax.imshow(img_show, cmap="gray")
    ax.set_title(f"t={step}" if step > 0 else "t=0 (输出)", fontsize=8)
    ax.axis("off")

plt.suptitle("反向去噪过程：纯噪声 → 生成图像", fontsize=14, y=1.05)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "denoising_trajectory.png", dpi=100, bbox_inches="tight")
plt.show()

---
## Section 7 · DDIM 加速采样（加分项）

DDPM 需要完整的 $T=1000$ 步采样，每步都要跑一次 U-Net，非常慢。

**DDIM（Denoising Diffusion Implicit Models）** 通过**确定性采样**只需 $S \ll T$ 步即可生成：

$$x_{t-1} = \sqrt{\bar{\alpha}_{t-1}} \cdot \hat{x}_0 + \sqrt{1 - \bar{\alpha}_{t-1}} \cdot \epsilon_\theta(x_t, t)$$

其中 $\hat{x}_0 = \frac{x_t - \sqrt{1-\bar{\alpha}_t} \cdot \epsilon_\theta}{\sqrt{\bar{\alpha}_t}}$ 是当前预测的"干净图像"。

> 直观理解：DDIM 跳步采样——比如只用 50 步而非 1000 步——因为确定性公式不需要每步加噪声。

In [ ]:
# ── DDIM 采样 ─────────────────────────────────────────
@torch.no_grad()
def ddim_sample(model, shape, ddim_steps=50, init_noise=None):
    """
    DDIM 加速采样。
    
    Args:
        model:      U-Net 去噪网络
        shape:      生成图像的形状 (B, C, H, W)
        ddim_steps: 采样步数（默认 50，比 DDPM 快 20 倍）
        init_noise: 可选的起始噪声（用于潜空间插值），None 则随机生成
    Returns:
        生成的图像
    """
    model.eval()
    
    # 均匀选取时间步子序列（保持在 CPU，用于索引 schedule 张量）
    step_indices = torch.linspace(T - 1, 0, ddim_steps, dtype=torch.long)
    
    # 使用给定的噪声或随机生成
    if init_noise is not None:
        x = init_noise.to(DEVICE)
    else:
        x = torch.randn(shape, device=DEVICE)
    
    for i in tqdm(range(ddim_steps), desc=f"DDIM-{ddim_steps}", leave=False):
        t = step_indices[i]
        t_batch = torch.full((shape[0],), t, device=DEVICE, dtype=torch.long)
        
        noise_pred = model(x, t_batch)
        
        # 预测 x_0（干净图像估计）—— 索引 CPU 上的 schedule 张量
        alpha_cumprod_t = alphas_cumprod[t].to(DEVICE)
        sqrt_alpha_cumprod_t = sqrt_alphas_cumprod[t].to(DEVICE)
        sqrt_om_alpha_cumprod_t = sqrt_one_minus_alphas_cumprod[t].to(DEVICE)
        
        x0_pred = (x - sqrt_om_alpha_cumprod_t * noise_pred) / sqrt_alpha_cumprod_t
        x0_pred = torch.clamp(x0_pred, -1, 1)  # 裁剪防止溢出
        
        # 确定下一步的时间索引
        if i < ddim_steps - 1:
            t_prev = step_indices[i + 1]
        else:
            t_prev = -1  # 最后一步到 t=-1（干净图像），Python int
        
        alpha_cumprod_prev = alphas_cumprod[t_prev].to(DEVICE) if t_prev >= 0 else torch.tensor(1.0, device=DEVICE)
        
        # DDIM 确定性更新
        x = (
            torch.sqrt(alpha_cumprod_prev) * x0_pred
            + torch.sqrt(1 - alpha_cumprod_prev) * noise_pred
        )
    
    return x


print("DDIM 采样函数已定义")
print(f"DDIM-50 步 vs DDPM-1000 步 → 速度提升 20×")

In [ ]:
# ── DDIM vs DDPM 对比 ─────────────────────────────────
import time

N_COMPARE = 16

print("DDIM-50 步采样中...")
t0 = time.time()
ddim_samples = ddim_sample(ema_model, shape=(N_COMPARE, 1, 28, 28), ddim_steps=50)
ddim_time = time.time() - t0
print(f"DDIM-50  耗时: {ddim_time:.1f}s")

print("DDPM-1000 步采样中...")
t0 = time.time()
ddpm_samples = ddpm_sample(ema_model, shape=(N_COMPARE, 1, 28, 28))
ddpm_time = time.time() - t0
print(f"DDPM-1000 耗时: {ddpm_time:.1f}s")

print(f"\n加速比: {ddpm_time / ddim_time:.1f}×")

# ── 并排对比 ──
fig, axes = plt.subplots(2, 1, figsize=(8, 10))

for ax, samples, title in [
    (axes[0], ddpm_samples, f"DDPM (1000 steps, {ddpm_time:.0f}s)"),
    (axes[1], ddim_samples, f"DDIM (50 steps, {ddim_time:.0f}s)"),
]:
    show = samples * 0.5 + 0.5
    show = torch.clamp(show, 0, 1)
    grid = make_grid(show, nrow=4, padding=2)
    ax.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray")
    ax.set_title(title, fontsize=13)
    ax.axis("off")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "ddim_vs_ddpm.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Section 8 · 潜在空间插值

扩散模型的一个有趣特性：在两个噪声向量之间做**球面线性插值（SLERP）**，生成的图像会平滑过渡。这说明模型学会了连续的潜在表示。

In [ ]:
# ── 潜在空间插值 ──────────────────────────────────────
@torch.no_grad()
def slerp(z1, z2, alpha):
    """球面线性插值（保持噪声方差，确保仍在数据流形附近）"""
    z1_flat = z1.reshape(z1.shape[0], -1)
    z2_flat = z2.reshape(z2.shape[0], -1)
    # 归一化到单位球面
    z1_norm = F.normalize(z1_flat, dim=-1)
    z2_norm = F.normalize(z2_flat, dim=-1)
    dot = (z1_norm * z2_norm).sum(dim=-1, keepdim=True)
    dot = torch.clamp(dot, -1.0, 1.0)
    theta = torch.acos(dot)
    sin_theta = torch.sin(theta)
    # 处理 theta≈0 的退化情况
    mask = sin_theta > 1e-6
    w1 = torch.where(mask, torch.sin((1 - alpha) * theta) / sin_theta, 1 - alpha)
    w2 = torch.where(mask, torch.sin(alpha * theta) / sin_theta, alpha)
    z_interp = w1 * z1_flat + w2 * z2_flat
    return z_interp.reshape(z1.shape)


print("执行潜在空间插值（SLERP）...")
z1 = torch.randn(1, 1, 28, 28, device=DEVICE)
z2 = torch.randn(1, 1, 28, 28, device=DEVICE)

interp_results = []
for i in range(8):
    alpha = i / 7.0
    z_interp = slerp(z1, z2, alpha)
    # 从插值后的噪声出发做 DDIM 去噪
    img = ddim_sample(ema_model, shape=(1, 1, 28, 28), ddim_steps=50, init_noise=z_interp)
    interp_results.append(img.cpu())

# ── 可视化 ──
fig, axes = plt.subplots(1, 8, figsize=(16, 2.5))
for i, img in enumerate(interp_results):
    img_show = img.squeeze() * 0.5 + 0.5
    axes[i].imshow(img_show, cmap="gray")
    axes[i].set_title(f"α={i/7:.2f}")
    axes[i].axis("off")

plt.suptitle("潜在空间插值 — 两噪声间 SLERP 连续过渡", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "latent_interpolation.png", dpi=100, bbox_inches="tight")
plt.show()

---
## 总结

### 本项目实现的核心内容

| 组件 | 说明 |
|------|------|
| **前向扩散** | 闭式解，$x_t = \sqrt{\bar{\alpha}_t} x_0 + \sqrt{1-\bar{\alpha}_t} \epsilon$ |
| **U-Net** | 编码器-解码器 + 跳跃连接 + Sinusoidal 时间嵌入 + 自注意力 |
| **训练** | MSE 预测噪声 + AdamW + EMA 平滑 + 混合精度 |
| **DDPM 采样** | 1000 步迭代去噪，随机过程 |
| **DDIM 采样** | 50 步确定性跳步，20× 加速 |
| **潜空间插值** | SLERP 连续过渡 |

### 未实现（Stable Diffusion 额外组件）

| 组件 | 作用 |
|------|------|
| **VAE** | 将 512×512 图像压缩到 64×64 潜空间，降低计算量 64× |
| **CLIP Text Encoder** | 文本 Prompt → 条件向量，实现文生图 |
| **Cross-Attention** | U-Net 中注入文本条件 |
| **Classifier-Free Guidance** | 训练时随机丢弃条件，推理时增强文本控制力 |

### 关键原理回顾

> Diffusion 模型 = 学会「从噪声中恢复数据」的神经网络。
> 训练时：向数据加噪 → 预测噪声 → 计算 L2 Loss。
> 推理时：从纯噪声开始 → 逐步预测并去除噪声 → 生成新数据。

---

*Build with PyTorch · DDPM (Ho et al., 2020) · DDIM (Song et al., 2021)*